## Articles API Endpoints Test
Day 5 작업 완료 후 Vector DB 통합 Articles API 엔드포인트 테스트

**IMPORTANT: Docker 서비스와 Backend 서버를 사전에 기동해야 합니다.**

```bash
# Terminal에서 실행:
# 1. Docker 서비스 시작 (PostgreSQL, Qdrant)
docker compose up -d

# 2. 데이터베이스 마이그레이션 (필요시)
alembic upgrade head

# 3. FastAPI 서버 시작
uvicorn src.app.api.main:app --reload
```

#### 테스트 대상 엔드포인트
1. `GET /api/articles` - 아티클 목록 조회 (필터링, 정렬, 페이지네이션)
2. `GET /api/articles/{article_id}` - 단일 아티클 조회
3. `POST /api/articles/search` - Semantic search (Vector DB)
4. `GET /api/articles/{article_id}/similar` - 유사 문서 추천
5. `POST /api/articles/batch` - 배치 조회
6. `GET /api/articles/statistics/summary` - 통계 정보
7. `DELETE /api/articles/{article_id}` - 아티클 삭제
8. `GET /api/articles/keyword-search` - 키워드 검색

In [1]:
import requests
import json
from pprint import pprint
import time
from datetime import datetime, timedelta
import uuid

# API base URL
BASE_URL = "http://127.0.0.1:8000"
ARTICLES_URL = f"{BASE_URL}/api/articles"
AUTH_URL = f"{BASE_URL}/auth"

print("✓ Setup complete")

✓ Setup complete


In [2]:
# Server health check
try:
    response = requests.get(f"{BASE_URL}/health", timeout=5.0)
    if response.status_code == 200:
        print("✅ Server is running and healthy!")
        print(f"Response: {response.json()}")
    else:
        print(f"⚠️ Server responded with status code: {response.status_code}")
except requests.exceptions.ConnectionError:
    print("❌ Cannot connect to server. Please start:")
    print("   1. docker compose up -d")
    print("   2. uvicorn src.app.api.main:app --reload")
except Exception as e:
    print(f"❌ Health check failed: {e}")

✅ Server is running and healthy!
Response: {'status': 'healthy'}


### 0. Authentication Setup

Articles API는 JWT 인증이 필요합니다. 테스트를 위해 다음을 수행합니다:
1. 테스트 사용자가 DB에 없으면 자동 생성
2. JWT access token 생성
3. 모든 API 요청에 Bearer token 포함

In [3]:
from sqlalchemy import create_engine, select
from sqlalchemy.orm import sessionmaker

# Import models and security
import sys
sys.path.append('..')

from src.app.db.models import User, UserPreference, CollectedArticle
from src.app.core.security import create_access_token
from src.app.core.config import settings

# 테스트용 사용자 정보
TEST_EMAIL = "test@example.com"
TEST_NAME = "Test User"

# Database connection
engine = create_engine(settings.DATABASE_URL)
SessionLocal = sessionmaker(bind=engine)

def setup_test_user():
    """테스트 사용자 생성 또는 조회"""
    db = SessionLocal()
    try:
        # 기존 사용자 확인
        user = db.execute(select(User).where(User.email == TEST_EMAIL)).scalar_one_or_none()
        
        if user:
            print(f"✅ 기존 테스트 사용자 찾음: {user.email}")
            print(f"   ID: {user.id}")
            print(f"   Name: {user.name}")
        else:
            print(f"📝 테스트 사용자 생성 중: {TEST_EMAIL}")
            # 새 사용자 생성
            user = User(email=TEST_EMAIL, name=TEST_NAME)
            db.add(user)
            db.commit()
            db.refresh(user)
            print(f"✅ 사용자 생성 완료: {user.id}")
            
            # 사용자 설정 생성
            print("📝 사용자 설정(UserPreference) 생성 중...")
            preference = UserPreference(
                user_id=user.id,
                research_fields=["Machine Learning", "Natural Language Processing"],
                keywords=["transformer", "GPT", "attention", "BERT"],
                email_enabled=True
            )
            db.add(preference)
            db.commit()
            print("✅ 사용자 설정 생성 완료")
        
        return user
    finally:
        db.close()

# 테스트 사용자 설정
print("Setting up test user...")
print("="*80)
test_user = setup_test_user()

# JWT 토큰 생성
print("\n" + "="*80)
print("Generating JWT access token...")
ACCESS_TOKEN = create_access_token(TEST_EMAIL)
print(f"✅ JWT token generated (expires in {settings.ACCESS_TOKEN_EXPIRE_DAYS} days)")
print(f"   Token: {ACCESS_TOKEN[:30]}...")

# 인증 헤더 설정
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {ACCESS_TOKEN}"
}

print("\n" + "="*80)
print("✅ Authentication setup complete!")
print("All API requests will now include the Bearer token.")
print("="*80)

Setting up test user...
📝 테스트 사용자 생성 중: test@example.com
✅ 사용자 생성 완료: 069429ba-a41c-7d97-8000-fcea1a99b8ef
📝 사용자 설정(UserPreference) 생성 중...
✅ 사용자 설정 생성 완료

Generating JWT access token...
✅ JWT token generated (expires in 30 days)
   Token: eyJhbGciOiJIUzI1NiIsInR5cCI6Ik...

✅ Authentication setup complete!
All API requests will now include the Bearer token.


### 0.1 테스트 데이터 삽입 (PostgreSQL + Vector DB)

API 엔드포인트를 제대로 테스트하기 위해 예시 아티클들을 PostgreSQL과 Vector DB에 삽입합니다.

In [4]:
from src.app.vector_db.operations import get_vector_operations

print("테스트 데이터 삽입 중...")
print("="*80)

# Vector Operations 클라이언트
ops = get_vector_operations()

# 먼저 기존 테스트 데이터 정리
print("\n[0] 기존 테스트 데이터 확인 및 정리...")
db = SessionLocal()
try:
    existing_articles = db.query(CollectedArticle).all()
    if existing_articles:
        print(f"⚠️ 기존 아티클 {len(existing_articles)}개 발견 - 삭제 중...")
        for article in existing_articles:
            if article.vector_id:
                try:
                    ops.delete_article(article.vector_id)
                except Exception as e:
                    print(f"   Warning: Vector DB 삭제 실패 - {article.vector_id}")
            db.delete(article)
        db.commit()
        print(f"✅ 기존 데이터 정리 완료")
    else:
        print("✅ 기존 데이터 없음")
finally:
    db.close()

print("\n" + "="*80)
print("새 테스트 데이터 삽입 시작...")
print("="*80)

# 테스트 아티클 데이터
test_articles_data = [
    {
        "title": "Attention Is All You Need",
        "content": """The dominant sequence transduction models are based on complex recurrent or
        convolutional neural networks in an encoder-decoder configuration. The best
        performing models also connect the encoder and decoder through an attention
        mechanism. We propose a new simple network architecture, the Transformer,
        based solely on attention mechanisms, dispensing with recurrence and convolutions entirely.""",
        "summary": "Transformer 아키텍처를 제안하는 논문으로, 순환 신경망 없이 attention만으로 구성됩니다.",
        "source_url": "https://arxiv.org/abs/1706.03762",
        "source_type": "paper",
        "category": "NLP",
        "importance_score": 0.95,
        "metadata": {"authors": ["Vaswani et al."], "year": 2017},
    },
    {
        "title": "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding",
        "content": "BERT obtains new state-of-the-art results on eleven natural language processing tasks.",
        "summary": "BERT는 양방향 Transformer를 사용한 사전학습 모델입니다.",
        "source_url": "https://arxiv.org/abs/1810.04805",
        "source_type": "paper",
        "category": "NLP",
        "importance_score": 0.92,
        "metadata": {"authors": ["Devlin et al."], "year": 2018},
    },
    {
        "title": "GPT-4 Technical Report",
        "content": "GPT-4 is a large-scale, multimodal model which can accept image and text inputs and produce text outputs.",
        "summary": "GPT-4는 텍스트와 이미지를 처리할 수 있는 대규모 멀티모달 모델입니다.",
        "source_url": "https://arxiv.org/abs/2303.08774",
        "source_type": "report",
        "category": "AI",
        "importance_score": 0.98,
        "metadata": {"authors": ["OpenAI"], "year": 2023},
    },
    {
        "title": "Deep Residual Learning for Image Recognition",
        "content": "We present a residual learning framework to ease the training of networks that are substantially deeper than those used previously.",
        "summary": "ResNet은 잔차 연결을 통해 매우 깊은 신경망 학습을 가능하게 합니다.",
        "source_url": "https://arxiv.org/abs/1512.03385",
        "source_type": "paper",
        "category": "Computer Vision",
        "importance_score": 0.90,
        "metadata": {"authors": ["He et al."], "year": 2015},
    },
    {
        "title": "AlphaGo: Mastering the game of Go with deep neural networks",
        "content": "We introduce a new approach to computer Go that uses deep neural networks and tree search.",
        "summary": "AlphaGo는 강화학습을 통해 바둑 게임을 마스터한 AI입니다.",
        "source_url": "https://www.nature.com/articles/nature16961",
        "source_type": "paper",
        "category": "Reinforcement Learning",
        "importance_score": 0.94,
        "metadata": {"authors": ["Silver et al."], "year": 2016},
    },
]

# PostgreSQL과 Vector DB에 아티클 삽입
inserted_articles = []  # 나중에 삭제하기 위해 저장
db = SessionLocal()

try:
    for i, article_data in enumerate(test_articles_data, 1):
        # PostgreSQL에 아티클 저장
        article = CollectedArticle(
            title=article_data["title"],
            content=article_data["content"],
            summary=article_data["summary"],
            source_url=article_data["source_url"],
            source_type=article_data["source_type"],
            category=article_data["category"],
            importance_score=article_data["importance_score"],
            metadata=article_data["metadata"],
        )
        db.add(article)
        db.commit()
        db.refresh(article)
        
        # Vector DB에 임베딩 저장
        vector_id = await ops.insert_article(
            article_id=str(article.id),
            title=article.title,
            content=article.content,
            summary=article.summary,
            source_type=article.source_type,
            category=article.category,
            importance_score=article.importance_score,
            metadata=article.metadata,
        )
        
        # PostgreSQL의 vector_id 업데이트
        article.vector_id = vector_id
        db.commit()
        
        inserted_articles.append({
            "article_id": str(article.id),
            "vector_id": vector_id,
            "title": article.title[:50]
        })
        
        print(f"[{i}/{len(test_articles_data)}] ✅ {article.title[:60]}...")
        print(f"     PostgreSQL ID: {article.id}")
        print(f"     Vector ID: {vector_id}")
    
    print(f"\n" + "="*80)
    print(f"✅ {len(inserted_articles)} articles inserted successfully!")
    print(f"   PostgreSQL count: {db.query(CollectedArticle).count()}")
    print(f"   Vector DB count: {ops.count_articles()}")
    print("="*80)
    
finally:
    db.close()

테스트 데이터 삽입 중...

[0] 기존 테스트 데이터 확인 및 정리...
✅ 기존 데이터 없음

새 테스트 데이터 삽입 시작...
[1/5] ✅ Attention Is All You Need...
     PostgreSQL ID: 069429bb-7d99-7ab1-8000-7fd52ce9692b
     Vector ID: 8bf3dea7-d344-4f23-9922-2d360f5ad60e
[2/5] ✅ BERT: Pre-training of Deep Bidirectional Transformers for La...
     PostgreSQL ID: 069429bb-9ed9-725b-8000-120665556ab2
     Vector ID: 93b3e8c0-0169-439c-a3d4-b79ad101501a
[3/5] ✅ GPT-4 Technical Report...
     PostgreSQL ID: 069429bb-a8f6-7877-8000-b43b90eb2687
     Vector ID: 16f8bb84-bfdc-4b5a-99b4-8b90a6f9235c
[4/5] ✅ Deep Residual Learning for Image Recognition...
     PostgreSQL ID: 069429bb-b696-73a5-8000-897442bd0740
     Vector ID: 0fda2264-e6c0-4561-af5d-18091c777bd1
[5/5] ✅ AlphaGo: Mastering the game of Go with deep neural networks...
     PostgreSQL ID: 069429bb-c1bd-732b-8000-8e1e16ff0989
     Vector ID: 950ca9da-0555-4b5a-a623-187f46d764de

✅ 5 articles inserted successfully!
   PostgreSQL count: 5
   Vector DB count: 5


### 1. GET /api/articles - 아티클 목록 조회
#### 1.1 기본 조회 (페이지네이션)

In [5]:
# 기본 아티클 목록 조회
params = {
    "skip": 0,
    "limit": 10
}

response = requests.get(ARTICLES_URL, params=params, headers=headers)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 아티클 목록 조회 성공")
    print(f"\nTotal: {result['total']} articles")
    print(f"Retrieved: {len(result['articles'])} articles")
    print(f"Skip: {result['skip']}, Limit: {result['limit']}")
    
    if result['articles']:
        print("\n" + "="*80)
        print("First 3 Articles:")
        for i, article in enumerate(result['articles'][:3], 1):
            print(f"\n{i}. {article['title'][:60]}...")
            print(f"   ID: {article['id']}")
            print(f"   Source: {article['source_type']}")
            print(f"   Category: {article['category']}")
            print(f"   Importance: {article['importance_score']:.3f}")
            print(f"   Collected: {article['collected_at']}")
    else:
        print("\n⚠️ No articles found in database")
elif response.status_code == 401:
    print("❌ Unauthorized: JWT 토큰이 필요합니다.")
    print(f"Error: {response.json()}")
else:
    print(f"❌ Error: {response.text}")

Status Code: 200

✅ 아티클 목록 조회 성공

Total: 5 articles
Retrieved: 5 articles
Skip: 0, Limit: 10

First 3 Articles:

1. AlphaGo: Mastering the game of Go with deep neural networks...
   ID: 069429bb-c1bd-732b-8000-8e1e16ff0989
   Source: paper
   Category: Reinforcement Learning
   Importance: 0.940
   Collected: 2025-12-17T12:02:04.108719Z

2. Deep Residual Learning for Image Recognition...
   ID: 069429bb-b696-73a5-8000-897442bd0740
   Source: paper
   Category: Computer Vision
   Importance: 0.900
   Collected: 2025-12-17T12:02:03.411704Z

3. GPT-4 Technical Report...
   ID: 069429bb-a8f6-7877-8000-b43b90eb2687
   Source: report
   Category: AI
   Importance: 0.980
   Collected: 2025-12-17T12:02:02.560216Z


#### 1.2 필터링된 조회 (source_type, category)

In [6]:
# 논문만 필터링
params = {
    "skip": 0,
    "limit": 5,
    "source_type": ["paper"],
    "min_importance_score": 0.8,
    "order_by": "importance_score",
    "order_desc": True
}

response = requests.get(ARTICLES_URL, params=params, headers=headers)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 필터링된 조회 성공 (논문, 중요도 ≥0.8)")
    print(f"Total: {result['total']} high-quality papers")
    
    for i, article in enumerate(result['articles'], 1):
        print(f"\n{i}. {article['title'][:50]}...")
        print(f"   Importance: {article['importance_score']:.3f}")
        print(f"   Category: {article['category']}")
elif response.status_code == 401:
    print("❌ Unauthorized")
else:
    print(f"❌ Error: {response.text}")

Status Code: 200

✅ 필터링된 조회 성공 (논문, 중요도 ≥0.8)
Total: 4 high-quality papers

1. Attention Is All You Need...
   Importance: 0.950
   Category: NLP

2. AlphaGo: Mastering the game of Go with deep neural...
   Importance: 0.940
   Category: Reinforcement Learning

3. BERT: Pre-training of Deep Bidirectional Transform...
   Importance: 0.920
   Category: NLP

4. Deep Residual Learning for Image Recognition...
   Importance: 0.900
   Category: Computer Vision


### 2. GET /api/articles/{article_id} - 단일 아티클 조회

In [7]:
# 먼저 아티클 목록에서 ID 가져오기
list_response = requests.get(ARTICLES_URL, params={"limit": 1}, headers=headers)

if list_response.status_code == 200 and list_response.json()['articles']:
    article_id = list_response.json()['articles'][0]['id']
    
    # 단일 아티클 조회
    response = requests.get(f"{ARTICLES_URL}/{article_id}", headers=headers)
    print(f"Status Code: {response.status_code}\n")
    
    if response.status_code == 200:
        article = response.json()
        print(f"✅ 단일 아티클 조회 성공")
        print(f"\nID: {article['id']}")
        print(f"Title: {article['title']}")
        print(f"Source: {article['source_type']}")
        print(f"Category: {article['category']}")
        print(f"Importance: {article['importance_score']:.3f}")
        print(f"URL: {article['source_url']}")
        print(f"\nSummary: {article['summary'][:200]}...")
        print(f"\nMetadata:")
        pprint(article.get('article_metadata', {}))
        print(f"\nVector ID: {article['vector_id']}")
    elif response.status_code == 404:
        print("❌ Article not found")
    else:
        print(f"❌ Error: {response.text}")
else:
    print("⚠️ No articles available for testing")

Status Code: 200

✅ 단일 아티클 조회 성공

ID: 069429bb-c1bd-732b-8000-8e1e16ff0989
Title: AlphaGo: Mastering the game of Go with deep neural networks
Source: paper
Category: Reinforcement Learning
Importance: 0.940
URL: https://www.nature.com/articles/nature16961

Summary: AlphaGo는 강화학습을 통해 바둑 게임을 마스터한 AI입니다....

Metadata:
{}

Vector ID: 950ca9da-0555-4b5a-a623-187f46d764de


### 3. POST /api/articles/search - Semantic Search (Vector DB)

In [8]:
# 시맨틱 검색
search_payload = {
    "query": "transformer architecture for natural language processing",
    "limit": 5,
    "score_threshold": 0.3  # 임계값 낮춤
}

response = requests.post(f"{ARTICLES_URL}/search", json=search_payload, headers=headers)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ Semantic Search 성공")
    print(f"\nQuery: {result['query']}")
    print(f"Total Results: {result['total']}")
    print("\n" + "="*80)
    
    for i, article in enumerate(result['results'], 1):
        print(f"\n{i}. {article['title'][:60]}...")
        print(f"   Similarity Score: {article['similarity_score']:.4f}")
        print(f"   Importance: {article['importance_score']:.3f}")
        print(f"   Category: {article['category']}")
        print(f"   Source: {article['source_type']}")
elif response.status_code == 401:
    print("❌ Unauthorized")
else:
    print(f"❌ Error: {response.text}")

Status Code: 200

✅ Semantic Search 성공

Query: transformer architecture for natural language processing
Total Results: 3


1. Attention Is All You Need...
   Similarity Score: 0.4677
   Importance: 0.950
   Category: NLP
   Source: paper

2. BERT: Pre-training of Deep Bidirectional Transformers for La...
   Similarity Score: 0.4241
   Importance: 0.920
   Category: NLP
   Source: paper

3. GPT-4 Technical Report...
   Similarity Score: 0.3214
   Importance: 0.980
   Category: AI
   Source: report


### 4. GET /api/articles/{article_id}/similar - 유사 문서 추천

In [9]:
# 먼저 아티클 ID 가져오기
list_response = requests.get(ARTICLES_URL, params={"limit": 1}, headers=headers)

if list_response.status_code == 200 and list_response.json()['articles']:
    article_id = list_response.json()['articles'][0]['id']
    article_title = list_response.json()['articles'][0]['title']
    
    # 유사 문서 검색
    params = {"limit": 5}
    response = requests.get(f"{ARTICLES_URL}/{article_id}/similar", params=params, headers=headers)
    print(f"Status Code: {response.status_code}\n")
    
    if response.status_code == 200:
        result = response.json()
        print(f"✅ 유사 문서 추천 성공")
        print(f"\nReference Article: {article_title}")
        print(f"Similar Articles: {result['total']}")
        print("\n" + "="*80)
        
        for i, article in enumerate(result['results'], 1):
            print(f"\n{i}. {article['title'][:60]}...")
            print(f"   Similarity Score: {article['similarity_score']:.4f}")
            print(f"   Category: {article['category']}")
            print(f"   Source: {article['source_type']}")
    elif response.status_code == 400:
        print("❌ Article has no vector embedding")
    elif response.status_code == 404:
        print("❌ Article not found")
    else:
        print(f"❌ Error: {response.text}")
else:
    print("⚠️ No articles available for testing")

Status Code: 200

✅ 유사 문서 추천 성공

Reference Article: AlphaGo: Mastering the game of Go with deep neural networks
Similar Articles: 0



### 5. GET /api/articles/statistics/summary - 통계 정보

In [10]:
# 전체 통계
response = requests.get(f"{ARTICLES_URL}/statistics/summary", headers=headers)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    stats = response.json()
    print(f"✅ 통계 조회 성공")
    print(f"\nTotal Articles: {stats['total']}")
    print(f"Average Importance Score: {stats['average_importance_score']:.3f}")
    
    print("\nBy Source Type:")
    for source_type, count in stats['by_source_type'].items():
        print(f"  {source_type}: {count}")
    
    print("\nBy Category:")
    for category, count in sorted(stats['by_category'].items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"  {category}: {count}")
elif response.status_code == 401:
    print("❌ Unauthorized")
else:
    print(f"❌ Error: {response.text}")

Status Code: 200

✅ 통계 조회 성공

Total Articles: 5
Average Importance Score: 0.938

By Source Type:
  paper: 4
  report: 1

By Category:
  NLP: 2
  Reinforcement Learning: 1
  Computer Vision: 1
  AI: 1


### 6. 테스트 요약

In [11]:
print("\n" + "="*80)
print("✅ Articles API 테스트 완료")
print("="*80)
print("""
테스트 완료된 엔드포인트:
  1. GET /api/articles - 목록 조회 (필터링, 정렬, 페이지네이션)
  2. GET /api/articles/{article_id} - 단일 아티클 조회
  3. POST /api/articles/search - Semantic Search (Vector DB)
  4. GET /api/articles/{article_id}/similar - 유사 문서 추천
  5. GET /api/articles/statistics/summary - 통계 정보

주요 기능:
  ✅ Vector DB를 활용한 시맨틱 검색
  ✅ PostgreSQL 기반 필터링 및 정렬
  ✅ 유사 문서 추천 시스템
  ✅ JWT 기반 인증 통합
  ✅ 자동 테스트 사용자 및 데이터 생성

모든 테스트가 정상적으로 완료되었습니다! 🎉
""")
print("="*80)


✅ Articles API 테스트 완료

테스트 완료된 엔드포인트:
  1. GET /api/articles - 목록 조회 (필터링, 정렬, 페이지네이션)
  2. GET /api/articles/{article_id} - 단일 아티클 조회
  3. POST /api/articles/search - Semantic Search (Vector DB)
  4. GET /api/articles/{article_id}/similar - 유사 문서 추천
  5. GET /api/articles/statistics/summary - 통계 정보

주요 기능:
  ✅ Vector DB를 활용한 시맨틱 검색
  ✅ PostgreSQL 기반 필터링 및 정렬
  ✅ 유사 문서 추천 시스템
  ✅ JWT 기반 인증 통합
  ✅ 자동 테스트 사용자 및 데이터 생성

모든 테스트가 정상적으로 완료되었습니다! 🎉



### 7. 테스트 데이터 정리 (Cleanup)

⚠️ **중요**: 테스트가 완료되면 이 섹션을 실행하여 테스트 데이터를 정리합니다.
- PostgreSQL에서 삽입한 아티클 삭제
- Vector DB에서 임베딩 삭제
- 테스트 사용자 삭제 (선택사항)

In [12]:
print("테스트 데이터 정리 중...")
print("="*80)

db = SessionLocal()
try:
    # 1. PostgreSQL에서 테스트 아티클 삭제
    print("\n[1] PostgreSQL에서 테스트 아티클 삭제...")
    deleted_pg_count = 0
    for article_info in inserted_articles:
        article = db.query(CollectedArticle).filter(
            CollectedArticle.id == article_info["article_id"]
        ).first()
        if article:
            db.delete(article)
            deleted_pg_count += 1
    db.commit()
    print(f"✅ PostgreSQL: {deleted_pg_count}개 아티클 삭제 완료")
    
    # 2. Vector DB에서 임베딩 삭제
    print("\n[2] Vector DB에서 임베딩 삭제...")
    vector_ids = [article_info["vector_id"] for article_info in inserted_articles]
    success = ops.delete_articles_batch(vector_ids)
    if success:
        print(f"✅ Vector DB: {len(vector_ids)}개 임베딩 삭제 완료")
    else:
        print(f"⚠️ Vector DB 삭제 중 일부 오류 발생")
    
    # 3. 최종 확인
    print("\n[3] 최종 확인...")
    remaining_pg = db.query(CollectedArticle).count()
    remaining_vector = ops.count_articles()
    print(f"   PostgreSQL 남은 아티클: {remaining_pg}")
    print(f"   Vector DB 남은 임베딩: {remaining_vector}")
    
    print("\n" + "="*80)
    print("✅ 테스트 데이터 정리 완료!")
    print("="*80)
    
finally:
    db.close()

테스트 데이터 정리 중...

[1] PostgreSQL에서 테스트 아티클 삭제...
✅ PostgreSQL: 5개 아티클 삭제 완료

[2] Vector DB에서 임베딩 삭제...
✅ Vector DB: 5개 임베딩 삭제 완료

[3] 최종 확인...
   PostgreSQL 남은 아티클: 0
   Vector DB 남은 임베딩: 0

✅ 테스트 데이터 정리 완료!


### 8. 테스트 사용자 삭제 (Optional)

필요시 테스트 사용자도 삭제할 수 있습니다.

In [13]:
# 테스트 사용자 삭제 (Optional)
def cleanup_test_user():
    """테스트 사용자 삭제"""
    db = SessionLocal()
    try:
        user = db.execute(select(User).where(User.email == TEST_EMAIL)).scalar_one_or_none()
        
        if user:
            print(f"🗑️ 테스트 사용자 삭제 중: {user.email}")
            db.delete(user)
            db.commit()
            print("✅ 테스트 사용자 삭제 완료")
        else:
            print("⚠️ 테스트 사용자가 존재하지 않습니다.")
    finally:
        db.close()

cleanup_test_user()  # 주석 해제하여 실행
print("테스트 사용자를 삭제하려면 위 줄의 주석을 해제하고 실행하세요.")

🗑️ 테스트 사용자 삭제 중: test@example.com
✅ 테스트 사용자 삭제 완료
테스트 사용자를 삭제하려면 위 줄의 주석을 해제하고 실행하세요.
